# 01 — Exploratory Data Analysis

Goal: understand the data before modeling. Write findings to `memory/findings.md`.

Checklist (update as you go):
- [ ] shapes, dtypes, memory
- [ ] target distribution / class balance
- [ ] missingness per column
- [ ] numeric distributions + outliers
- [ ] categorical cardinalities
- [ ] correlations / mutual information with target
- [ ] train vs test distribution drift

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.utils import load_train, load_test, load_sample_submission, PATHS

In [3]:
df = load_train()
df.shape

(630000, 21)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   id                       630000 non-null  int64   
 1   Soil_Type                630000 non-null  category
 2   Soil_pH                  630000 non-null  float64 
 3   Soil_Moisture            630000 non-null  float64 
 4   Organic_Carbon           630000 non-null  float64 
 5   Electrical_Conductivity  630000 non-null  float64 
 6   Temperature_C            630000 non-null  float64 
 7   Humidity                 630000 non-null  float64 
 8   Rainfall_mm              630000 non-null  float64 
 9   Sunlight_Hours           630000 non-null  float64 
 10  Wind_Speed_kmh           630000 non-null  float64 
 11  Crop_Type                630000 non-null  category
 12  Crop_Growth_Stage        630000 non-null  category
 13  Season                   630000 non-null  ca

In [9]:
df.describe()

,id,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,Previous_Irrigation_mm
count,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000
mean,314999.500000,6.482497,37.304482,0.922858,1.744605,26.998166,61.563180,1462.207566,7.513382,10.375394,7.517745,62.318177
std,181865.479132,0.922504,16.377082,0.365808,0.952321,8.623621,19.708152,612.989738,1.999322,5.689458,4.218124,34.246939
min,0.000000,4.800000,8.000000,0.300000,0.100000,12.000000,25.000000,0.380000,4.000000,0.500000,0.300000,0.020000
25%,157499.750000,5.690000,23.340000,0.610000,0.930000,19.517500,45.390000,954.570000,5.760000,5.280000,3.880000,33.140000
50%,314999.500000,6.440000,37.750000,0.910000,1.740000,26.960000,61.650000,1467.160000,7.580000,10.480000,7.380000,61.150000
75%,472499.250000,7.270000,51.270000,1.220000,2.580000,34.540000,79.120000,2054.280000,9.250000,15.430000,11.140000,92.690000
max,629999.000000,8.200000,64.990000,1.600000,3.500000,42.000000,94.990000,2499.690000,11.000000,20.000000,15.000000,119.990000


In [7]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
df[cat_cols] = df[cat_cols].astype("category")

for col in cat_cols:
    print(f"{col}: {sorted(df[col].cat.categories.tolist())}")

Soil_Type: ['Clay', 'Loamy', 'Sandy', 'Silt']
Crop_Type: ['Cotton', 'Maize', 'Potato', 'Rice', 'Sugarcane', 'Wheat']
Crop_Growth_Stage: ['Flowering', 'Harvest', 'Sowing', 'Vegetative']
Season: ['Kharif', 'Rabi', 'Zaid']
Irrigation_Type: ['Canal', 'Drip', 'Rainfed', 'Sprinkler']
Water_Source: ['Groundwater', 'Rainwater', 'Reservoir', 'River']
Mulching_Used: ['No', 'Yes']
Region: ['Central', 'East', 'North', 'South', 'West']
Irrigation_Need: ['High', 'Low', 'Medium']


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = df.select_dtypes(include="number").corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax)
ax.set_title("Correlation Heatmap (numeric features)")
plt.tight_layout()
plt.show()

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.model_selection import StratifiedKFold

# ── paths & data ─────────────────────────────────────────────────────────────
DATA_DIR = Path.cwd().parent / "data"
df_train = pd.read_csv(DATA_DIR / "train.csv")

NUM_COLS = [
    "Soil_Moisture", "Temperature_C", "Rainfall_mm", "Wind_Speed_kmh",
    "Humidity_pct", "Sunlight_Hours", "Crop_Water_Need_mm",
    "Field_Area_hectares", "Elevation_m", "Soil_pH", "Fertilizer_Used_kg",
]
CAT_COLS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]
TARGET = "Irrigation_Need"
CLASSES = ["Low", "Medium", "High"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# ── encode target ─────────────────────────────────────────────────────────────
X = df_train[NUM_COLS + CAT_COLS].copy()
for c in CAT_COLS:
    X[c] = X[c].astype("category")
y = df_train[TARGET].map(CLASS_TO_IDX).to_numpy()

# ── class weights (inverse frequency, mean=1) ────────────────────────────────
counts = np.bincount(y, minlength=3)
class_w = len(y) / (3 * counts)
sample_w = class_w[y]

# ── 5-fold stratified CV ──────────────────────────────────────────────────────
lgbm_params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "lambda_l2": 1.0,
    "verbose": -1,
    "seed": 42,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros((len(X), 3), dtype=np.float32)

print("Training 5-fold LightGBM (Experiment #001 config)…")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    w_tr, w_va = sample_w[tr_idx], sample_w[va_idx]

    dtr = lgb.Dataset(X_tr, label=y_tr, weight=w_tr, categorical_feature=CAT_COLS)
    dva = lgb.Dataset(X_va, label=y_va, weight=w_va, categorical_feature=CAT_COLS, reference=dtr)

    model = lgb.train(
        lgbm_params, dtr,
        num_boost_round=2000,
        valid_sets=[dva],
        valid_names=["val"],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    preds = model.predict(X_va, num_iteration=model.best_iteration)
    oof[va_idx] = preds
    fold_score = balanced_accuracy_score(y_va, preds.argmax(axis=1))
    print(f"  fold {fold+1}/5: BalAcc={fold_score:.5f}  best_iter={model.best_iteration}")

oof_score = balanced_accuracy_score(y, oof.argmax(axis=1))
per_class = recall_score(y, oof.argmax(axis=1), average=None)
print(f"\nOOF BalAcc (argmax): {oof_score:.5f}")
print(f"Per-class recall  — Low: {per_class[0]:.4f}  Medium: {per_class[1]:.4f}  High: {per_class[2]:.4f}")

# ── post-hoc per-class log-offset tuning ────────────────────────────────────
eps   = 1e-12
logp  = np.log(oof + eps)
offsets = np.zeros(3)
best_v  = balanced_accuracy_score(y, logp.argmax(axis=1))
grid    = np.linspace(-1.5, 1.5, 31)

for _ in range(3):
    improved = False
    for k in range(3):
        base = offsets.copy()
        for g in grid:
            base[k] = g
            v = balanced_accuracy_score(y, (logp + base).argmax(axis=1))
            if v > best_v:
                best_v, offsets, improved = v, base.copy(), True
    if not improved:
        break

print(f"Tuned BalAcc      : {best_v:.5f}  (offsets {offsets.round(1).tolist()})")